# HW1 Part 1 - Sherlock Holmes Continued Pre-training with QLoRA

Clean cluster notebook for the RTX 3090 run. The notebook is controlled by `hw1_config.json`.

Default mode is **submission replay**: it reloads the saved metrics/logs and regenerates all tables and figures without retraining. Set `CONFIG["submission"]["run_training"] = True` only when launching a new training run.


## 0. Cluster Start Command

Start Jupyter from the login node with the RTX 3090 course allocation:

```bash
sjupyter --time 0-1:00:00 --qos course --part rtx3090 --gpu rtx_3090:1 --account cours_ibm26
```

For a new training run, request more time, for example `--time 0-5:00:00`.


In [ ]:
!hostname
!which nvidia-smi
!nvidia-smi


## 1. Load Config, Explain Run Names, and Select Experiment

The JSON config is the single source of truth. The abbreviations in the run names are expanded here before the rest of the notebook uses them.


In [ ]:
import copy
import json
import os
from pathlib import Path
import pandas as pd

NOTEBOOK_VARIANT = "cluster"
CONFIG_PATH = Path(os.environ.get("HW1_CONFIG_PATH", "hw1_config.json"))
CONFIG = json.loads(CONFIG_PATH.read_text())

# Cluster paths for the uploaded data and saved experiment outputs.
CONFIG["environment"]["cluster"]["data_dir"] = "/home/guykalat/HW1-second_try/Data/drive-download-20260512T211012Z-3-001"
CONFIG["environment"]["cluster"]["output_root"] = "runs"

# Keep dependency installation manual on the cluster so a working CUDA/PyTorch env is not overwritten.
CONFIG["environment"]["install_dependencies"] = False

# Submission replay mode:
# False = do not load/train the model; regenerate tables and figures from saved result folders.
# True  = train the active experiment below.
RUN_TRAINING = bool(CONFIG.get("submission", {}).get("run_training", False))
RUN_GRID = bool(CONFIG["experiments"].get("run_grid", False))

ACTIVE_EXPERIMENT = CONFIG["experiments"].get("active", "lr_1e4_r16_attn_mlp")
# To train a different single run, edit the JSON config or uncomment one line:
# ACTIVE_EXPERIMENT = "rank_r8_lr2e4_attn_mlp"
# ACTIVE_EXPERIMENT = "rank_r16_lr2e4_attn_mlp"
# ACTIVE_EXPERIMENT = "attn_only_r16_lr2e4"
# ACTIVE_EXPERIMENT = "len512_r16_lr2e4_attn_mlp"


def deep_update(base, updates):
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


experiment_by_name = {exp["name"]: exp for exp in CONFIG["experiments"]["experiments"]}
if ACTIVE_EXPERIMENT not in experiment_by_name:
    raise ValueError(f"Unknown experiment {ACTIVE_EXPERIMENT}. Available: {list(experiment_by_name)}")

ACTIVE_EXPERIMENT_CONFIG = experiment_by_name[ACTIVE_EXPERIMENT]
deep_update(CONFIG, copy.deepcopy(ACTIVE_EXPERIMENT_CONFIG.get("overrides", {})))
CONFIG["experiments"]["active"] = ACTIVE_EXPERIMENT
CONFIG["training"]["run_name"] = ACTIVE_EXPERIMENT_CONFIG.get("overrides", {}).get("training", {}).get("run_name", ACTIVE_EXPERIMENT)

EXPERIMENT_LABELS = {
    exp["name"]: exp.get("display_name", exp["name"])
    for exp in CONFIG["experiments"]["experiments"]
}
EXPERIMENT_SHORT_LABELS = {
    exp["name"]: exp.get("short_name", exp.get("display_name", exp["name"]))
    for exp in CONFIG["experiments"]["experiments"]
}

experiment_overview = pd.DataFrame([
    {
        "run_name": exp["name"],
        "display_name": exp.get("display_name", exp["name"]),
        "category": exp.get("category", ""),
        "meaning": exp.get("hypothesis", ""),
    }
    for exp in CONFIG["experiments"]["experiments"]
])

print("Loaded config from", CONFIG_PATH.resolve())
print("Submission replay mode:", not RUN_TRAINING)
print("Run full ablation grid:", RUN_GRID)
print("Active experiment:", ACTIVE_EXPERIMENT)
print("Active display name:", EXPERIMENT_LABELS.get(ACTIVE_EXPERIMENT, ACTIVE_EXPERIMENT))
print("Active model:", CONFIG["model"]["model_id"])
print("Run name:", CONFIG["training"]["run_name"])
print("Data dir:", CONFIG["environment"]["cluster"]["data_dir"])
print("Output root:", CONFIG["environment"]["cluster"]["output_root"])
print("Slurm hint:", CONFIG["environment"]["cluster"].get("slurm_hint"))
display(experiment_overview)


## 2. Dependency Policy and Environment Check

On the cluster, do not reinstall packages every run. First make sure the current kernel sees a compatible GPU stack. If this check fails, fix the environment before model loading.


In [ ]:
import subprocess
import sys

if CONFIG["environment"].get("install_dependencies", False):
    packages = CONFIG["environment"]["packages"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])
else:
    print("Dependency installation skipped by config. This is intentional on the cluster.")


In [ ]:
import torch
print("torch:", torch.__version__)
print("torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())


## 3. Imports and Paths

These imports power the rest of the pipeline. PEFT adds LoRA adapters, TRL provides the SFTTrainer, and bitsandbytes is used only when QLoRA 4-bit loading is enabled.


In [ ]:
import gc
import math
import random
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer

set_seed(CONFIG["project"]["seed"])
random.seed(CONFIG["project"]["seed"])

variant_cfg = CONFIG["environment"][NOTEBOOK_VARIANT]
DATA_DIR = Path(os.environ.get("HW1_DATA_DIR", variant_cfg["data_dir"])).expanduser()
OUTPUT_ROOT = Path(os.environ.get("HW1_OUTPUT_ROOT", variant_cfg["output_root"])).expanduser()
RUN_NAME = CONFIG["training"]["run_name"]
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())


## 4. Memory Estimates

The assignment asks for model-size intuition before choosing a model. These are estimates, not measured allocations. They explain why QLoRA is useful: full fine-tuning stores weights, gradients, optimizer state, and activations, while QLoRA keeps frozen base weights in 4-bit and trains small adapters.


In [ ]:
def estimate_activation_gb(layers, hidden_size, seq_len, micro_batch, dtype_bytes=2, checkpointing=True):
    multiplier = CONFIG["memory_estimates"]["activation_multiplier_with_checkpointing"] if checkpointing else CONFIG["memory_estimates"]["activation_multiplier_without_checkpointing"]
    return layers * hidden_size * seq_len * micro_batch * dtype_bytes * multiplier / 1024**3

arch = {1: {"layers": 24, "hidden": 2048}, 3: {"layers": 36, "hidden": 2048}, 8: {"layers": 40, "hidden": 4096}, 13: {"layers": 40, "hidden": 5120}}
rows=[]
for size_b in CONFIG["memory_estimates"]["model_sizes_b"]:
    params=size_b*1_000_000_000
    weights_fp16=params*2/1024**3
    weights_4bit=params*0.5/1024**3
    full_static=params*CONFIG["memory_estimates"]["assumed_training_bytes_per_param_full"]/1024**3
    qlora_static=params*CONFIG["memory_estimates"]["assumed_training_bytes_per_param_qlora"]/1024**3
    for seq_len in CONFIG["memory_estimates"]["sequence_lengths"]:
        acts=estimate_activation_gb(arch[size_b]["layers"], arch[size_b]["hidden"], seq_len, CONFIG["memory_estimates"]["micro_batch_size"], checkpointing=CONFIG["training"]["gradient_checkpointing"])
        rows.append({"model_size_B":size_b,"max_length":seq_len,"weights_fp16_GB":round(weights_fp16,2),"weights_4bit_GB":round(weights_4bit,2),"full_train_static_GB":round(full_static,2),"qlora_static_GB":round(qlora_static,2),"activation_est_GB":round(acts,2),"qlora_total_est_GB":round(qlora_static+acts,2)})
memory_df=pd.DataFrame(rows)
display(memory_df)


## 5. Tokenizer

The tokenizer defines what a “token” means. All corpus token counts and max-length decisions should be measured with the same tokenizer used by the model.


In [ ]:
def torch_dtype_from_config(value):
    if value in (None, "auto"):
        return "auto"
    return getattr(torch, value)

model_id = CONFIG["model"]["model_id"]
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=CONFIG["model"].get("trust_remote_code", False))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print("Tokenizer loaded:", model_id)
print("Tokenizer max length:", tokenizer.model_max_length)
print("Training max_seq_length:", CONFIG["data"]["max_seq_length"])


## 6. Data Cleaning and Token Counting

This cell reads the nine Project Gutenberg files, strips boilerplate, normalizes text, splits into coherent chapters/stories, and counts model-tokenized corpus size.


In [ ]:
START_RE = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.IGNORECASE | re.DOTALL)
END_RE = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*", re.IGNORECASE | re.DOTALL)
SECTION_RE = re.compile(r"(?m)^\s*((?:CHAPTER|Chapter)\s+[IVXLCDM0-9]+[^\n]*|(?:PART|Part)\s+[IVXLCDM0-9]+[^\n]*|[IVXLCDM]{1,8}\.?\s*)$")


def clean_gutenberg_text(text):
    cfg = CONFIG["data"]["cleaning"]
    text = text.replace("\ufeff", "")
    if cfg.get("strip_gutenberg_header_footer", True):
        start = START_RE.search(text)
        if start:
            text = text[start.end():]
        end = END_RE.search(text)
        if end:
            text = text[:end.start()]
    if cfg.get("normalize_unicode"):
        text = unicodedata.normalize(cfg["normalize_unicode"], text)
    if cfg.get("normalize_newlines", True):
        text = text.replace("\r\n", "\n").replace("\r", "\n")
    if cfg.get("collapse_many_blank_lines", True):
        text = re.sub(r"\n{4,}", "\n\n\n", text)
    return text.strip()


def split_sections(book_id, title, text):
    matches = list(SECTION_RE.finditer(text))
    sections=[]
    if len(matches) < 2:
        return [{"book_id":book_id,"title":title,"section":"full_book","text":text}]
    for i, match in enumerate(matches):
        start=match.start(); end=matches[i+1].start() if i+1 < len(matches) else len(text)
        section_text=text[start:end].strip()
        if len(section_text) >= CONFIG["data"]["min_section_chars"]:
            sections.append({"book_id":book_id,"title":title,"section":match.group(1).strip(),"text":section_text})
    return sections or [{"book_id":book_id,"title":title,"section":"full_book","text":text}]

missing=[]
raw_books=[]
sections=[]
for book in CONFIG["data"]["books"]:
    path = DATA_DIR / book["file"]
    if not path.exists():
        missing.append(str(path))
        continue
    text = path.read_text(encoding="utf-8", errors="replace")
    cleaned = clean_gutenberg_text(text)
    raw_books.append({**book, "chars_cleaned": len(cleaned), "text": cleaned})
    sections.extend(split_sections(book["id"], book["title"], cleaned))
if missing:
    raise FileNotFoundError("Missing Gutenberg files:\n" + "\n".join(missing[:20]) + f"\nDATA_DIR={DATA_DIR}")

section_lengths=[]
for s in tqdm(sections, desc="Tokenizing coherent sections"):
    section_lengths.append(len(tokenizer(s["text"], add_special_tokens=False)["input_ids"]))
section_df=pd.DataFrame([{k:v for k,v in s.items() if k != "text"} for s in sections])
section_df["tokens"]=section_lengths
print("Books loaded:", len(raw_books))
print("Coherent sections:", len(sections))
print("Total corpus tokens:", int(section_df["tokens"].sum()))
display(section_df.groupby(["book_id","title"], as_index=False)["tokens"].sum())


## 7. Validation Split and Chunking

The assignment asks for about 5% held-out validation, ideally whole chapters/stories. This cell holds out coherent sections from the end of the canon, then chunks train/validation text into windows that fit the configured context length.


In [ ]:
def select_validation_sections(sections, lengths, fraction):
    target=max(1, int(sum(lengths)*fraction))
    chosen=set(); running=0
    for idx in range(len(sections)-1, -1, -1):
        chosen.add(idx); running += lengths[idx]
        if running >= target:
            break
    return chosen, running, target


def chunk_sections(selected_sections, tokenizer, max_length, stride, append_eos=True):
    chunks=[]; meta=[]; eos=tokenizer.eos_token or ""
    for s in selected_sections:
        text=s["text"] + (eos if append_eos else "")
        ids=tokenizer(text, add_special_tokens=False)["input_ids"]
        step=stride if stride and stride > 0 else max_length
        for start in range(0, len(ids), step):
            window=ids[start:start+max_length]
            if len(window) < 32:
                continue
            chunks.append(tokenizer.decode(window))
            meta.append({"book_id":s["book_id"],"title":s["title"],"section":s["section"],"tokens":len(window)})
            if start + max_length >= len(ids):
                break
    return chunks, meta

val_indices, val_tokens, target_tokens = select_validation_sections(sections, section_lengths, CONFIG["data"]["validation_fraction"])
train_sections=[s for i,s in enumerate(sections) if i not in val_indices]
val_sections=[s for i,s in enumerate(sections) if i in val_indices]
train_chunks, train_meta = chunk_sections(train_sections, tokenizer, CONFIG["data"]["max_seq_length"], CONFIG["data"]["chunk_stride"], CONFIG["data"].get("append_eos_between_sections", True))
val_chunks, val_meta = chunk_sections(val_sections, tokenizer, CONFIG["data"]["max_seq_length"], CONFIG["data"]["chunk_stride"], CONFIG["data"].get("append_eos_between_sections", True))
train_dataset=Dataset.from_dict({"text": train_chunks})
eval_dataset=Dataset.from_dict({"text": val_chunks})

print(f"Validation target tokens: {target_tokens:,}; selected coherent validation tokens: {val_tokens:,}")
print("Train chunks:", len(train_dataset), "Eval chunks:", len(eval_dataset))
print("Held-out validation sections:")
display(pd.DataFrame([{k:v for k,v in s.items() if k != "text"} for s in val_sections]))
length_df=pd.DataFrame(train_meta + val_meta)
length_df["split"]=["train"]*len(train_meta)+["validation"]*len(val_meta)
display(length_df.groupby("split")["tokens"].describe())
length_df.hist(column="tokens", by="split", bins=30, figsize=(10,4))
plt.suptitle("Input length histogram after chunking")
plt.show()


## 8. Data Figures for Report

This cell saves the corpus/input-length figures used in the report. It reuses `section_df` and `length_df` from the preprocessing and chunking cells, so it does not reload the model or repeat training.



In [ ]:
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 1) Input-length histogram after chunking. This uses the real model tokenizer,
# because length_df was built from tokenizer-produced chunk lengths.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, split in zip(axes, ["train", "validation"]):
    values = length_df.loc[length_df["split"] == split, "tokens"]
    ax.hist(values, bins=30, color="#2E74B5", edgecolor="white", alpha=0.9)
    ax.axvline(CONFIG["data"]["max_seq_length"], color="#B23A48", linestyle="--", linewidth=1.5)
    ax.set_title(f"{split.capitalize()} chunks")
    ax.set_xlabel("Tokens per chunk")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Number of chunks")
fig.suptitle("Input length histogram after chunking")
fig.tight_layout()
hist_path = FIGURE_DIR / "input_length_histogram_by_split.png"
fig.savefig(hist_path, bbox_inches="tight", dpi=220)
plt.show()
print("Saved", hist_path)

# 2) Token count by book. This answers the corpus-size part of the assignment visually.
book_tokens = section_df.groupby(["book_id", "title"], as_index=False)["tokens"].sum()
book_tokens = book_tokens.sort_values("tokens", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(book_tokens["title"], book_tokens["tokens"], color="#2E74B5")
ax.set_xlabel("Tokens")
ax.set_title("Cleaned corpus token count by book")
ax.grid(axis="x", alpha=0.25)
for i, v in enumerate(book_tokens["tokens"]):
    ax.text(v + max(book_tokens["tokens"]) * 0.01, i, f"{int(v):,}", va="center", fontsize=8)
fig.tight_layout()
book_path = FIGURE_DIR / "token_count_by_book.png"
fig.savefig(book_path, bbox_inches="tight", dpi=220)
plt.show()
print("Saved", book_path)

# 3) Save the table behind the figures for reproducibility.
length_df.to_csv(FIGURE_DIR / "input_lengths_by_split.csv", index=False)
book_tokens.to_csv(FIGURE_DIR / "token_count_by_book.csv", index=False)
print("Saved", FIGURE_DIR / "input_lengths_by_split.csv")
print("Saved", FIGURE_DIR / "token_count_by_book.csv")



## 9. Full Ablation Grid: Model Loading, LoRA/QLoRA, Training, and Saving

This cell runs every configured ablation one after another. Each run gets its own folder under `runs/`, including the adapter, metrics summary, README, and training log.


In [ ]:
RUN_GRID = bool(CONFIG["experiments"].get("run_grid", False))


def resolve_compute_dtype(cfg):
    requested = cfg["quantization"].get("bnb_4bit_compute_dtype", "auto")
    if requested == "auto":
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return getattr(torch, requested)


def available_linear_suffixes(model):
    suffixes = set()
    linear_types = (torch.nn.Linear,)
    try:
        import bitsandbytes as bnb
        linear_types = (torch.nn.Linear, bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)
    except Exception:
        pass
    for name, module in model.named_modules():
        if isinstance(module, linear_types):
            suffixes.add(name.split(".")[-1])
    return sorted(suffixes)


def filtered_target_modules(model, requested):
    if requested == "all-linear":
        return "all-linear"
    available = set(available_linear_suffixes(model))
    keep = [m for m in requested if m in available]
    if not keep:
        print("No requested LoRA targets matched; falling back to all-linear")
        return "all-linear"
    missing = [m for m in requested if m not in available]
    if missing:
        print("Missing requested target module suffixes:", missing)
    print("LoRA target modules:", keep)
    return keep


def build_peft_config(cfg, target_modules):
    return LoraConfig(
        r=cfg["lora"]["r"],
        lora_alpha=cfg["lora"]["lora_alpha"],
        lora_dropout=cfg["lora"]["lora_dropout"],
        bias=cfg["lora"]["bias"],
        task_type=cfg["lora"]["task_type"],
        target_modules=target_modules,
    )


def make_experiment_config(base_cfg, exp):
    exp_cfg = copy.deepcopy(base_cfg)
    deep_update(exp_cfg, copy.deepcopy(exp.get("overrides", {})))
    exp_cfg["experiments"]["active"] = exp["name"]
    exp_cfg["training"]["run_name"] = exp.get("overrides", {}).get("training", {}).get("run_name", exp["name"])
    return exp_cfg


def load_model_for_training(cfg):
    model_id = cfg["model"]["model_id"]
    quantization_config = None
    if cfg["quantization"].get("load_in_4bit", False):
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=resolve_compute_dtype(cfg),
            bnb_4bit_use_double_quant=cfg["quantization"].get("bnb_4bit_use_double_quant", True),
            bnb_4bit_quant_type=cfg["quantization"].get("bnb_4bit_quant_type", "nf4"),
        )

    model_kwargs = dict(
        device_map=cfg["model"].get("device_map", "auto"),
        quantization_config=quantization_config,
        trust_remote_code=cfg["model"].get("trust_remote_code", False),
        attn_implementation=cfg["model"].get("attn_implementation", "sdpa"),
    )
    model_kwargs["dtype"] = torch_dtype_from_config(cfg["model"].get("torch_dtype", "auto"))
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    except TypeError:
        model_kwargs["torch_dtype"] = model_kwargs.pop("dtype")
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

    model.config.use_cache = False
    if cfg["quantization"].get("load_in_4bit", False):
        model = prepare_model_for_kbit_training(model)
    if cfg["training"].get("gradient_checkpointing", True):
        model.gradient_checkpointing_enable()
    target_modules = filtered_target_modules(model, cfg["lora"]["target_modules"])
    peft_config = build_peft_config(cfg, target_modules)
    return model, peft_config


def create_sft_config(cfg, output_dir):
    requested_dtype = cfg["model"].get("torch_dtype", "float16")
    use_bf16 = requested_dtype == "bfloat16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16
    kwargs = dict(
        output_dir=str(output_dir),
        run_name=cfg["training"]["run_name"],
        max_length=cfg["data"]["max_seq_length"],
        dataset_text_field="text",
        packing=cfg["training"].get("packing", False),
        num_train_epochs=cfg["training"]["num_train_epochs"],
        max_steps=cfg["training"].get("max_steps", -1),
        per_device_train_batch_size=cfg["training"]["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["training"]["per_device_eval_batch_size"],
        gradient_accumulation_steps=cfg["training"]["gradient_accumulation_steps"],
        learning_rate=cfg["training"]["learning_rate"],
        lr_scheduler_type=cfg["training"]["lr_scheduler_type"],
        warmup_ratio=cfg["training"]["warmup_ratio"],
        optim=cfg["training"]["optim"],
        weight_decay=cfg["training"].get("weight_decay", 0.0),
        max_grad_norm=cfg["training"].get("max_grad_norm", 1.0),
        logging_steps=cfg["training"]["logging_steps"],
        eval_strategy="steps",
        eval_steps=cfg["training"]["eval_steps"],
        save_strategy="steps",
        save_steps=cfg["training"]["save_steps"],
        save_total_limit=cfg["training"]["save_total_limit"],
        gradient_checkpointing=cfg["training"].get("gradient_checkpointing", True),
        bf16=use_bf16,
        fp16=use_fp16,
        report_to=cfg["training"].get("report_to", ["tensorboard"]),
        seed=cfg["project"]["seed"],
        remove_unused_columns=True,
    )
    if cfg["training"].get("loss_type"):
        kwargs["loss_type"] = cfg["training"]["loss_type"]
    try:
        return SFTConfig(**kwargs)
    except TypeError as exc:
        print("Retrying SFTConfig without newer optional args:", exc)
        kwargs.pop("loss_type", None)
        return SFTConfig(**kwargs)


def perplexity_from_loss(loss):
    return float(math.exp(loss)) if loss is not None and loss < 20 else float("inf")


def trainer_perplexity(trainer, label):
    metrics = trainer.evaluate()
    metrics[f"{label}_perplexity"] = perplexity_from_loss(metrics.get("eval_loss"))
    print(label, metrics)
    return metrics


def sliding_window_perplexity(model, tokenizer, text, max_length, stride, max_eval_tokens=None):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    input_ids = enc.input_ids[:, :max_eval_tokens] if max_eval_tokens else enc.input_ids
    seq_len = input_ids.size(1)
    device = getattr(model, "device", None) or next(model.parameters()).device
    nll_sum = 0.0
    n_tokens = 0
    prev_end_loc = 0
    for begin_loc in tqdm(range(0, seq_len, stride), desc="Sliding-window PPL"):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_window = input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_window.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = model(input_window, labels=target_ids)
        valid_tokens = (target_ids != -100).sum().item()
        loss_tokens = valid_tokens - target_ids.size(0)
        nll_sum += outputs.loss.item() * loss_tokens
        n_tokens += loss_tokens
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
    return math.exp(nll_sum / max(n_tokens, 1))


def build_datasets_for_config(cfg):
    train_chunks_exp, train_meta_exp = chunk_sections(
        train_sections,
        tokenizer,
        cfg["data"]["max_seq_length"],
        cfg["data"]["chunk_stride"],
        cfg["data"].get("append_eos_between_sections", True),
    )
    val_chunks_exp, val_meta_exp = chunk_sections(
        val_sections,
        tokenizer,
        cfg["data"]["max_seq_length"],
        cfg["data"]["chunk_stride"],
        cfg["data"].get("append_eos_between_sections", True),
    )
    train_ds = Dataset.from_dict({"text": train_chunks_exp})
    eval_ds = Dataset.from_dict({"text": val_chunks_exp})
    return train_ds, eval_ds, train_meta_exp, val_meta_exp


def save_training_log(trainer, output_dir):
    log_df = pd.DataFrame(trainer.state.log_history)
    if not log_df.empty:
        log_df.to_csv(output_dir / "training_log.csv", index=False)
        print("Saved", output_dir / "training_log.csv")
    return log_df


def run_one_experiment(exp, base_cfg):
    exp_cfg = make_experiment_config(base_cfg, exp)
    run_name = exp_cfg["training"]["run_name"]
    output_dir = OUTPUT_ROOT / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("RUNNING:", run_name)
    print("DISPLAY:", exp.get("display_name", run_name))
    print("HYPOTHESIS:", exp.get("hypothesis", ""))
    print("=" * 100)

    set_seed(exp_cfg["project"]["seed"])
    train_ds, eval_ds, train_meta_exp, val_meta_exp = build_datasets_for_config(exp_cfg)
    print("Train chunks:", len(train_ds), "Eval chunks:", len(eval_ds))
    print("max_seq_length:", exp_cfg["data"]["max_seq_length"], "stride:", exp_cfg["data"]["chunk_stride"])
    print("LoRA r:", exp_cfg["lora"]["r"], "alpha:", exp_cfg["lora"]["lora_alpha"], "dropout:", exp_cfg["lora"]["lora_dropout"])

    model, peft_config = load_model_for_training(exp_cfg)
    training_args = create_sft_config(exp_cfg, output_dir)
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    if hasattr(trainer.model, "print_trainable_parameters"):
        trainer.model.print_trainable_parameters()

    val_text_exp = "\n\n".join(s["text"] for s in val_sections)
    before_metrics = trainer_perplexity(trainer, "before_training")
    before_sw_ppl = sliding_window_perplexity(
        trainer.model,
        tokenizer,
        val_text_exp,
        exp_cfg["data"]["max_seq_length"],
        exp_cfg["evaluation"]["sliding_stride"],
        exp_cfg["evaluation"].get("max_eval_tokens"),
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    trainer_stats = trainer.train(resume_from_checkpoint=exp_cfg["training"].get("resume_from_checkpoint"))
    print(trainer_stats)
    peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3 if torch.cuda.is_available() else None
    if peak_reserved_gb is not None:
        print(f"Peak reserved GPU memory: {peak_reserved_gb:.2f} GB")

    adapter_dir = output_dir / "final_adapter"
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print("Saved adapter/tokenizer to", adapter_dir)

    after_metrics = trainer_perplexity(trainer, "after_training")
    after_sw_ppl = sliding_window_perplexity(
        trainer.model,
        tokenizer,
        val_text_exp,
        exp_cfg["data"]["max_seq_length"],
        exp_cfg["evaluation"]["sliding_stride"],
        exp_cfg["evaluation"].get("max_eval_tokens"),
    )
    log_df = save_training_log(trainer, output_dir)

    before_ppl = before_metrics["before_training_perplexity"]
    after_ppl = after_metrics["after_training_perplexity"]
    improvement_pct = 100 * (before_ppl - after_ppl) / before_ppl
    results = {
        "experiment": {
            "name": exp["name"],
            "display_name": exp.get("display_name", exp["name"]),
            "short_name": exp.get("short_name", exp["name"]),
            "category": exp.get("category", ""),
            "hypothesis": exp.get("hypothesis", ""),
        },
        "config": exp_cfg,
        "before_perplexity": before_ppl,
        "after_perplexity": after_ppl,
        "improvement_pct": improvement_pct,
        "before_sliding_perplexity": before_sw_ppl,
        "after_sliding_perplexity": after_sw_ppl,
        "before_training": before_metrics,
        "after_training": after_metrics,
        "peak_reserved_gb": peak_reserved_gb,
        "num_train_chunks": len(train_ds),
        "num_eval_chunks": len(eval_ds),
        "trainable_log": str(output_dir / "training_log.csv") if not log_df.empty else None,
    }
    results_path = output_dir / "metrics_summary.json"
    results_path.write_text(json.dumps(results, indent=2, default=str))
    (output_dir / "README.md").write_text(
        f"# {exp.get('display_name', exp['name'])}\n\n"
        f"Run name: `{run_name}`\n\n"
        f"Hypothesis: {exp.get('hypothesis', '')}\n\n"
        f"Before PPL: {before_ppl:.4f}\n\n"
        f"After PPL: {after_ppl:.4f}\n\n"
        f"Improvement: {improvement_pct:.2f}%\n"
    )
    print("Saved metrics to", results_path)

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


if not RUN_TRAINING:
    model = None
    peft_config = None
    grid_results = []
    print("Submission replay mode: skipping model loading and training.")
elif RUN_GRID:
    model = None
    peft_config = None
    grid_results = []
    BASE_GRID_CONFIG = copy.deepcopy(CONFIG)
    for exp in CONFIG["experiments"]["experiments"]:
        grid_results.append(run_one_experiment(exp, BASE_GRID_CONFIG))

    grid_summary = pd.DataFrame([
        {
            "experiment": r["experiment"]["name"],
            "display_name": r["experiment"]["display_name"],
            "short_name": r["experiment"]["short_name"],
            "category": r["experiment"]["category"],
            "before_ppl": r["before_perplexity"],
            "after_ppl": r["after_perplexity"],
            "relative_improvement_%": r["improvement_pct"],
            "before_sliding_ppl": r["before_sliding_perplexity"],
            "after_sliding_ppl": r["after_sliding_perplexity"],
            "peak_reserved_gb": r["peak_reserved_gb"],
        }
        for r in grid_results
    ]).sort_values("after_ppl")
    grid_summary.to_csv(OUTPUT_ROOT / "assignment_grid_summary.csv", index=False)
    grid_summary.to_csv(OUTPUT_ROOT / "experiment_summary.csv", index=False)
    print("Saved", OUTPUT_ROOT / "assignment_grid_summary.csv")
    display(grid_summary)
else:
    model, peft_config = load_model_for_training(CONFIG)
    print(peft_config)


## 10. TRL SFT Trainer

In grid mode, trainer creation is performed inside Section 9 for each experiment.


In [ ]:
print('Grid mode: trainer creation was handled inside Section 9 for each experiment.')


## 11. Evaluation Before Training

In grid mode, before-training evaluation is performed inside Section 9 for each experiment.


In [ ]:
print('Grid mode: before-training evaluation was handled inside Section 9 for each experiment.')


## 12. Train Active Experiment

In grid mode, training is performed inside Section 9 for every ablation.


In [ ]:
print('Grid mode: training was handled inside Section 9 for each experiment.')


## 13. Save Adapter and Evaluate After Training

In grid mode, saving and after-training evaluation are performed inside Section 9 for every ablation.


In [ ]:
print('Grid mode: adapter saving and after-training evaluation were handled inside Section 9 for each experiment.')


## 14. Qualitative Sampling

For the final submission run, qualitative generation is optional. The quantitative ablation grid is the main result.


In [ ]:
print('Grid mode: qualitative sampling skipped to keep the full ablation run focused and shorter.')


## 15. Aggregate Experiment Results

This cell scans saved run folders and builds the comparison table used in the report.


In [ ]:
experiment_plan = pd.DataFrame([
    {
        "run_name": e["name"],
        "display_name": e.get("display_name", e["name"]),
        "category": e.get("category", ""),
        "hypothesis": e["hypothesis"],
        "overrides": json.dumps(e.get("overrides", {})),
    }
    for e in CONFIG["experiments"]["experiments"]
])
display(experiment_plan)


def metric_value(metrics, *paths):
    for path in paths:
        cur = metrics
        ok = True
        for key in path:
            if isinstance(cur, dict) and key in cur:
                cur = cur[key]
            else:
                ok = False
                break
        if ok:
            return cur
    return None


rows = []
for path in sorted(OUTPUT_ROOT.glob("*/metrics_summary.json")):
    try:
        m = json.loads(path.read_text())
        exp_name = (
            metric_value(m, ("experiment", "name"))
            or m.get("active_experiment")
            or path.parent.name
        )
        before_ppl = metric_value(
            m,
            ("before_perplexity",),
            ("before_training", "before_training_perplexity"),
        )
        after_ppl = metric_value(
            m,
            ("after_perplexity",),
            ("after_training", "after_training_perplexity"),
        )
        improvement_pct = metric_value(
            m,
            ("improvement_pct",),
            ("improvement", "eval_ppl_relative_percent"),
        )
        before_sw = metric_value(
            m,
            ("before_sliding_perplexity",),
            ("before_training", "sliding_window_perplexity"),
        )
        after_sw = metric_value(
            m,
            ("after_sliding_perplexity",),
            ("after_training", "sliding_window_perplexity"),
        )
        rows.append(
            {
                "run_dir": path.parent.name,
                "experiment": exp_name,
                "display_name": EXPERIMENT_LABELS.get(exp_name, exp_name),
                "short_name": EXPERIMENT_SHORT_LABELS.get(exp_name, exp_name),
                "category": metric_value(m, ("experiment", "category")) or "",
                "before_ppl": before_ppl,
                "after_ppl": after_ppl,
                "relative_improvement_%": improvement_pct,
                "before_sliding_ppl": before_sw,
                "after_sliding_ppl": after_sw,
                "metrics_path": str(path),
            }
        )
    except Exception as exc:
        print("Skipping", path, exc)

summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["after_ppl", "relative_improvement_%"], ascending=[True, False])
    summary_df.to_csv(OUTPUT_ROOT / "experiment_summary.csv", index=False)
    summary_df.to_csv(OUTPUT_ROOT / "assignment_grid_summary.csv", index=False)
    print("Saved", OUTPUT_ROOT / "experiment_summary.csv")
    print("Saved", OUTPUT_ROOT / "assignment_grid_summary.csv")
else:
    print("No metrics_summary.json files found under", OUTPUT_ROOT)
display(summary_df)


## 16. Result Figures and TensorBoard-Style Curves

This cell creates the report figures from saved metrics and training logs.


In [ ]:
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = OUTPUT_ROOT / "assignment_grid_summary.csv"
if "summary_df" in globals() and not summary_df.empty:
    plot_df = summary_df.copy()
elif summary_csv.exists():
    plot_df = pd.read_csv(summary_csv)
else:
    plot_df = pd.DataFrame()

if plot_df.empty:
    print("No metrics found yet. Run this cell after the experiment folders are saved.")
else:
    if "improvement_pct" in plot_df.columns and "relative_improvement_%" not in plot_df.columns:
        plot_df["relative_improvement_%"] = plot_df["improvement_pct"]
    if "display_name" not in plot_df.columns:
        plot_df["display_name"] = plot_df["experiment"].map(EXPERIMENT_LABELS).fillna(plot_df["experiment"])
    if "short_name" not in plot_df.columns:
        plot_df["short_name"] = plot_df["experiment"].map(EXPERIMENT_SHORT_LABELS).fillna(plot_df["experiment"])

    plot_df = plot_df.sort_values("after_ppl")
    labels = plot_df["short_name"].astype(str)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(plot_df))
    width = 0.36
    ax.bar([i - width/2 for i in x], plot_df["before_ppl"], width=width, label="Before", color="#A6B7C8")
    ax.bar([i + width/2 for i in x], plot_df["after_ppl"], width=width, label="After", color="#2E74B5")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Perplexity (lower is better)")
    ax.set_title("Validation perplexity before and after training")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    ppl_path = FIGURE_DIR / "perplexity_before_after.png"
    fig.savefig(ppl_path, bbox_inches="tight", dpi=220)
    plt.show()
    print("Saved", ppl_path)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(labels, plot_df["relative_improvement_%"], color="#2E74B5")
    ax.set_ylabel("Improvement (%)")
    ax.set_title("Relative perplexity improvement by experiment")
    ax.grid(axis="y", alpha=0.25)
    for i, v in enumerate(plot_df["relative_improvement_%"]):
        if pd.notna(v):
            ax.text(i, v + 0.2, f"{v:.2f}%", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    imp_path = FIGURE_DIR / "perplexity_improvement_by_experiment.png"
    fig.savefig(imp_path, bbox_inches="tight", dpi=220)
    plt.show()
    print("Saved", imp_path)

# TensorBoard-style curves from the saved trainer logs.
log_rows = []
for csv_path in sorted(OUTPUT_ROOT.glob("*/training_log.csv")):
    try:
        df = pd.read_csv(csv_path)
        run = csv_path.parent.name
        df["run_dir"] = run
        df["short_name"] = EXPERIMENT_SHORT_LABELS.get(run, run)
        log_rows.append(df)
    except Exception as exc:
        print("Could not read", csv_path, exc)

if not log_rows:
    print("No training_log.csv files found, so TensorBoard-style training curves were not generated.")
else:
    logs_df = pd.concat(log_rows, ignore_index=True)
    logs_df.to_csv(FIGURE_DIR / "combined_training_logs.csv", index=False)
    print("Saved", FIGURE_DIR / "combined_training_logs.csv")

    if "eval_loss" in logs_df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        for run, run_df in logs_df.dropna(subset=["eval_loss"]).groupby("run_dir"):
            label = EXPERIMENT_SHORT_LABELS.get(run, run)
            ax.plot(run_df["step"], run_df["eval_loss"], marker="o", linewidth=1.8, label=label)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Validation loss")
        ax.set_title("Validation loss during training")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False, fontsize=8)
        fig.tight_layout()
        eval_curve_path = FIGURE_DIR / "training_eval_loss_by_run.png"
        fig.savefig(eval_curve_path, bbox_inches="tight", dpi=220)
        plt.show()
        print("Saved", eval_curve_path)

    if "loss" in logs_df.columns:
        best_run = None
        if "plot_df" in globals() and not plot_df.empty:
            best_run = plot_df.sort_values("after_ppl").iloc[0]["run_dir"]
        if best_run is None:
            best_run = logs_df["run_dir"].iloc[0]
        best_logs = logs_df[logs_df["run_dir"] == best_run]
        fig, ax = plt.subplots(figsize=(10, 4.5))
        train_points = best_logs.dropna(subset=["loss"])
        if not train_points.empty:
            ax.plot(train_points["step"], train_points["loss"], marker="o", label="Training loss", color="#2E74B5")
        if "eval_loss" in best_logs.columns:
            eval_points = best_logs.dropna(subset=["eval_loss"])
            if not eval_points.empty:
                ax.plot(eval_points["step"], eval_points["eval_loss"], marker="s", label="Validation loss", color="#B23A48")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training curve for best run: {EXPERIMENT_SHORT_LABELS.get(best_run, best_run)}")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        fig.tight_layout()
        best_curve_path = FIGURE_DIR / "best_run_training_curve.png"
        fig.savefig(best_curve_path, bbox_inches="tight", dpi=220)
        plt.show()
        print("Saved", best_curve_path)


## 17. Optional: Cancel the Slurm Session

Use this only after saving/downloading the notebook and result files.


In [ ]:
# import os
# os.system("scancel " + os.environ["SLURM_JOBID"])
